In [ ]:
import os

import pickle
import h5py
import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import gaussian_filter1d

from src.utils.jax_utils.vis_utils import read_h5

from src.utils.jax_utils.vis_utils import pos_init_cartesian_2d
from src.utils.jax_utils.vis_utils import mls_2nd_order, energy_spectrum
from src.utils.data_utils import load_metadata

In [ ]:
def get_metadata(path):
    metadata = load_metadata(path)
    dim, dx = metadata["dim"], metadata["dx"]
    N = metadata["num_particles_max"]
    Nx = round(N ** (1 / dim))
    bounds = metadata["bounds"]
    box_size = np.array(bounds)[:, 1] - np.array(bounds)[:, 0]
    return dim, dx, N, Nx, box_size


def set_up_spectrum_fn(dim, dx, Nx, box_size):
    def comp_spectrum(r, u):
        assert r.shape == (Nx**2, dim)
        assert u.shape == (Nx**2, dim)

        u_mls = np.array(
            [
                mls_2nd_order(
                    r, r_grid, u[:, i], box_size, dx, dim, kernel_name="Quintic", h_factor=0.8
                )
                for i in range(dim)
            ]
        )
        spectrum = energy_spectrum(u_mls.reshape(dim, Nx, Nx))
        spectrum *= 1 / (4 * np.pi)  # undo normalization to match magnitude from PDF above
        return k_axis, spectrum

    k_axis = np.arange(1, Nx // 2 + 1)
    r_grid = pos_init_cartesian_2d(box_size, dx)
    return comp_spectrum


def plt_spectra(paths, names, step_max, every_n=1, spectrum_num_plts=2, save_suffix=""):
    dim, dx, N, Nx, box_size = get_metadata(paths[0])
    comp_spectrum = set_up_spectrum_fn(dim, dx, Nx, box_size)

    ekin_xaxis = np.arange(0, step_max + 1, 50 // every_n)
    ekin = []
    ekin_target = []
    spectra = []
    for i, path in enumerate(paths):
        # print("Start path", path)

        ekin_sub = []
        if "2D_KOLM_4096_200kevery1" in path:  # ML dataset
            # Ekin
            traj_0 = h5py.File(f"{path}/test.h5")["00000"]  # TODO: average over all 5 trajs
            for t in ekin_xaxis:
                u = traj_0["u"][t]
                ekin_sub.append((u**2).sum())
            # Spectrums
            r = traj_0["position"][t]  # last time step
        elif "SPH" in path:  # SPH data
            for t in ekin_xaxis:
                frame = read_h5(f"{path}/traj_{str(t * every_n).zfill(5)}.h5")
                u = frame["u"]
                ekin_sub.append((u**2).sum())
            r = frame["r"]
        elif "rollout" in path:
            with open(path, "rb") as f:  # from rollout
                rollout = pickle.load(f)
                for t in ekin_xaxis:
                    u = rollout["predicted_u_vel"][t]
                    u_gt = rollout["ground_truth_u_vel"][t]
                    ekin_sub.append((u**2).sum())
                    ekin_target.append((u_gt**2).sum())
                r = rollout["predicted_rollout"][t]
                # r_gt = rollout["ground_truth_rollout"][t]
        if i < spectrum_num_plts:
            # print(r.shape, u.shape)
            k_axis, spectrum = comp_spectrum(r, u)
            # print(len(spectrum), len(ekin_sub))
            spectra.append(spectrum[1 : len(k_axis) + 1])
        else:
            spectra.append(None)
        # print(ekin[0].shape, spectra[0].shape)
        ekin.append(0.5 * np.array(ekin_sub) * dx**2)

    font_size = 12
    plt.rcParams.update({"font.size": font_size})
    fig, axs = plt.subplots(1, 2, figsize=(10, 4))
    for i, (ek, sp) in enumerate(zip(ekin, spectra)):
        # dissip = gaussian_filter1d(-(ek[1:]-ek[:-1]), sigma=1.0)
        # axs[0].plot(ekin_xaxis[:-1], dissip, label=names[i])
        style = "-" if "Dataset" in names[i] else "--"
        axs[0].plot(ekin_xaxis, ek, style, label=names[i])
        if sp is not None:
            axs[1].plot(k_axis, gaussian_filter1d(sp, sigma=0.5), style, label=names[i])

    axs[0].set_xlabel("Time step")
    axs[0].set_ylabel(r"Kinetic energy  $E_{kin}$")
    # axs[0].set_ylabel(r"Dissipation rate  $- \partial E_{kin} / \partial t$")
    axs[0].set_xlim(0, ekin_xaxis[-1])

    axs[1].set_xlabel("Wavenumber k")
    axs[1].set_ylabel(f"Energy spectrum at step {step_max}")
    axs[1].set_xscale("log")
    axs[1].set_xlim(1, len(k_axis))
    axs[1].set_xticks([1, 10, 32])
    axs[1].set_xticklabels([f"{int(k)}" for k in axs[1].get_xticks()])

    for ax in axs:
        ax.grid()
    # axs[0].set_yscale('log')
    axs[0].set_ylim(100, 150)
    axs[1].set_yscale("log")
    axs[0].legend(loc="lower left")
    plt.tight_layout()
    plt.savefig(f"../logs/figs/spectra_{every_n}{save_suffix}.pdf")
    plt.savefig(f"../logs/figs/spectra_{every_n}{save_suffix}.png")
    plt.show()


def plt_fields(paths, names, time_steps, size_factor=1, every_n=1, save_suffix=""):
    font_size = 20 * size_factor
    plt.rcParams.update({"font.size": font_size})
    rows, cols = len(paths), len(time_steps)
    fig, axs = plt.subplots(
        rows,
        cols,
        figsize=(size_factor * 4 * cols, size_factor * 4 * rows),
        sharex=True,
        sharey=True,
    )

    for j, path in enumerate(paths):
        for i, t in enumerate(time_steps):
            if "2D_KOLM_4096_200kevery1" in path:  # ML dataset
                traj_0 = h5py.File(f"{path}/test.h5")["00000"]
                r, u = traj_0["position"][t], traj_0["u"][t]  # no time coarsening
            elif "SPH" in path:  # SPH data
                frame = read_h5(f"{path}/traj_{str(t * every_n).zfill(5)}.h5")
                r, u = frame["r"], frame["u"]
            elif "rollout" in path:
                with open(path, "rb") as f:  # from rollout
                    rollout = pickle.load(f)
                    u = rollout["predicted_u_vel"][t]
                    # u_gt = rollout['ground_truth_u_vel'][t]
                    r = rollout["predicted_rollout"][t]
                    # r_gt = rollout['ground_truth_rollout'][t]
            # print(r.shape, u.shape)
            u_abs = np.linalg.norm(u, axis=1)

            axs[j, i].scatter(r[:, 0], r[:, 1], c=u_abs, s=8, vmin=0, vmax=7, cmap="turbo")

    for i, ax in enumerate(axs[0]):
        ax.set_title(
            f"step={time_steps[i]} (t={time_steps[i] * 0.0005 * every_n}s)", fontsize=font_size
        )
    for i, ax in enumerate(axs[:, 0]):
        ax.set_ylabel(names[i])

    for ax in axs.flatten():
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_aspect("equal")
        ax.set_xlim(0, 2 * np.pi)
        ax.set_ylim(0, 2 * np.pi)
    plt.tight_layout()
    plt.savefig(f"../logs/figs/fields_{every_n}{save_suffix}.pdf")
    plt.savefig(f"../logs/figs/fields_{every_n}{save_suffix}.png", dpi=50)
    plt.show()

In [ ]:
def get_paths_names(rlt_type="rollouts_1100", is_every1=True, root_logs="../logs/paper", ind=0):
    if is_every1:
        ckpts = {  # on every 1 step
            "simple": "2025-02-08_02-46-56",
            "tvf": "2025-02-08_02-47-01",
            "simple_u": "2025-02-08_02-47-06",
            "simple_rlx": "2025-02-09_06-22-19",
        }
    else:
        ckpts = {  # on every 10 step
            "simple": "2025-02-09_17-04-48",
            "tvf": "2025-02-09_17-04-52",
            "simple_u": "2025-02-09_17-04-56",
            "simple_rlx": "2025-02-09_21-33-00",
        }

    def rlt_path(ckpt_date, rlt_type, ind):
        return os.path.join(root_logs, ckpt_date, rlt_type, f"rollout_{str(ind).zfill(4)}.pkl")

    paths = [
        "../data/2D_KOLM_4096_200kevery1",
        # "/home/atoshev/code/sph-turbulence/gen_dataset/data/2D_KOLM_SPH_0_20250210-232517_SPH_15",
        "../data/2D_KOLM_SPH_0_20250210-232340_TVF_15",
        rlt_path(ckpts["simple"], rlt_type, ind),
        rlt_path(ckpts["simple_rlx"], rlt_type, ind),
        rlt_path(ckpts["tvf"], rlt_type, ind),
        rlt_path(ckpts["simple_u"], rlt_type, ind),
    ]
    names = [
        "Dataset",
        # "SPH",
        "SPH",
        "Simple-Base",
        "TVF-Base",
        r"Simple-$\mathbf{u}$",
        "TVF-Rlx" if is_every1 else "TVF-Rlx-Closure",
    ]
    return paths, names


# check scripts/rollout.sh for generating rollouts

# files = os.listdir(path_sph)
# files = [f for f in files if f.endswith('.h5')]
# files.sort()
# print(len(files))
# print(files)

In [ ]:
paths, names = get_paths_names("rollouts_1100", True)
plt_fields(paths=paths, names=names, time_steps=[0, 20, 50, 1000], save_suffix="")

paths, names = get_paths_names("rollouts_1000_nsph", True)
plt_fields(paths=paths, names=names, time_steps=[0, 20, 50, 1000], save_suffix="_nsph")

In [ ]:
paths_, names_ = get_paths_names("rollouts_1100", True)
plt_spectra(paths=paths_, names=names_, step_max=1000, spectrum_num_plts=3, save_suffix="")

paths_, names_ = get_paths_names("rollouts_1000_nsph", True)
plt_spectra(paths=paths_, names=names_, step_max=1000, spectrum_num_plts=4, save_suffix="_nsph")

# Every 10

In [ ]:
paths, names = get_paths_names("rollouts_100", False)
plt_fields(paths=paths, names=names, time_steps=[0, 5, 10, 50], every_n=10)

paths, names = get_paths_names("rollouts_100_nsph", False)
plt_fields(paths=paths, names=names, time_steps=[0, 5, 10, 50], every_n=10, save_suffix="_nsph")

In [ ]:
paths_, names_ = get_paths_names("rollouts_100", False)
plt_spectra(paths=paths_, names=names_, step_max=50, every_n=10, spectrum_num_plts=3)

paths_, names_ = get_paths_names("rollouts_100_nsph", False)
plt_spectra(
    paths=paths_, names=names_, step_max=50, every_n=10, spectrum_num_plts=4, save_suffix="_nsph"
)